In [55]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.8.0


In [56]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [57]:
input_query =  inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

#### Attention Scores

In [58]:
# empty tensors
att_scores = torch.empty(inputs.shape[0])

for i , emb in enumerate(inputs):
    att_scores[i] = torch.dot(emb , input_query)


att_scores
    

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

#### Attention Weights

In [59]:
# get the normalization using softmax
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum()    

softmax_naive(att_scores)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [60]:
att_normalized =  torch.softmax(att_scores , dim=0)

In [61]:
# context vector = sum(attention_weights * embeddings)
ctx_vector = torch.sum(att_normalized.unsqueeze(1) * inputs, dim=0)


In [62]:
ctx_vector

tensor([0.4419, 0.6515, 0.5683])

Computing attention weights for all input tokens

In [63]:
inputs.shape

torch.Size([6, 3])

In [64]:
# empty tensors
att_scores = torch.empty(6,6)

for i , x_i in enumerate(inputs):
    for j , x_j in enumerate(inputs):
        att_scores[i,j] = torch.dot(x_i , x_j)


att_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [65]:
att_scores = inputs @ inputs.T
att_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [66]:
atn_weights = torch.softmax(att_scores , dim =1)
atn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [67]:
all_ctx_vecs =  atn_weights @ inputs
all_ctx_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

#### Attention Mechanism with trainable parameters

In [68]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [69]:
x_2 =  inputs[1]
d_in = inputs.shape[1]  # here is the input embedding size
d_out = 2

In [70]:
torch.manual_seed(123)

# create the query weight matrix
W_query =torch.nn.Parameter(torch.rand(d_in , d_out))
W_key =torch.nn.Parameter(torch.rand(d_in , d_out))
W_value =torch.nn.Parameter(torch.rand(d_in , d_out))

print(W_query, W_key , W_value)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True) Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True) Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)


In [71]:
# printing shapes for references
print(x_2)
print(x_2.shape , W_query.shape)

query_2 = x_2 @ W_query

print(query_2.shape , query_2)

tensor([0.5500, 0.8700, 0.6600])
torch.Size([3]) torch.Size([3, 2])
torch.Size([2]) tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)


In [72]:
# Here query for the considered token is the same and we need to calculate the keys and values for each token

# * Here keep in mide the query key and val are random weights and will be trained later
keys = inputs @ W_key
values = inputs @ W_value

keys , values

(tensor([[0.3669, 0.7646],
         [0.4433, 1.1419],
         [0.4361, 1.1156],
         [0.2408, 0.6706],
         [0.1827, 0.3292],
         [0.3275, 0.9642]], grad_fn=<MmBackward0>),
 tensor([[0.1855, 0.8812],
         [0.3951, 1.0037],
         [0.3879, 0.9831],
         [0.2393, 0.5493],
         [0.1492, 0.3346],
         [0.3221, 0.7863]], grad_fn=<MmBackward0>))

![image](./images/1.png)

In [73]:
keys_2 = keys[1]
att_score_22 = torch.dot(query_2 , keys_2)

att_score_22

tensor(1.8524, grad_fn=<DotBackward0>)

In [74]:
# Here we can calculate all at onece with the query2 (see the image above)

attns_scores_2 = query_2 @ keys.T

attns_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [75]:
d_k = keys.shape[1]

att_wts_2 = torch.softmax(attns_scores_2 / d_k**0.5 , dim=-1)

torch.sum(att_wts_2)

tensor(1., grad_fn=<SumBackward0>)

![image](./images/2.png)

In [76]:
context_vec_2 = att_wts_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)


Now we can ge the full contect vector for all the queries

In [77]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [78]:
# init the QKV
d_in ,d_out

(3, 2)

In [79]:
torch.manual_seed(123)

W_query =torch.nn.Parameter(torch.rand(d_in , d_out))
W_key =torch.nn.Parameter(torch.rand(d_in , d_out))
W_value =torch.nn.Parameter(torch.rand(d_in , d_out))

In [80]:
inputs.shape , W_query.shape 

(torch.Size([6, 3]), torch.Size([3, 2]))

In [81]:
# All matrixes

query = inputs @ W_query
key = inputs @ W_key
value = inputs @ W_value

In [82]:
d_k

2

In [83]:
class SelfAttention(): 
    def __init__(self , query , key ,value):
        self.query = query
        self.key = key
        self.value = value

    def calContextVec(self):

        # ctx_vector =  torch.empty() 

        att_matrix = self.query @ self.key.T / self.key.shape[-1]**0.5
        att_weights = torch.softmax(att_matrix , dim=-1)
        ctx_vector = att_weights @ self.value

        return ctx_vector
                    

In [84]:
self_attention =  SelfAttention(query , key , value)

In [85]:
ctx_vec = self_attention.calContextVec()

print(ctx_vec)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


#### Here are the thing that we need to think over my implementation
- Here all of the parameters in my code (q,k,v) are already defined so they are static and wont be trainable
- Here I have not defined the faward methode() this is a pytorch api wher that is optimized from optimizer.step() methode
- Not following the PyTorch best practice

In [86]:
# Correct implementation
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self , d_in , d_out):
        super().__init__() # Call the __init__() method of the parent class (nn.Module) before doing anything else
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    
    def forward(self ,x):

        keys =  x @ self.W_key
        values =  x @ self.W_value
        queries =  x @ self.W_query

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec


In [87]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [88]:
# Here is a more optimal version from using Linear Layers
class SelfAttention_v2(nn.Module):
    def __init__(self , d_in ,d_out ,qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out , bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out , bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out , bias=qkv_bias)

    def forward(self , x):

        queries = self.W_query(x)
        keys = self.W_key (x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec




In [89]:
self_att2 =  SelfAttention_v2(d_in , d_out , False)
ctx_vector_2 = self_att2(inputs)

ctx_vector_2

tensor([[0.5085, 0.3508],
        [0.5084, 0.3508],
        [0.5084, 0.3506],
        [0.5074, 0.3471],
        [0.5076, 0.3446],
        [0.5077, 0.3493]], grad_fn=<MmBackward0>)

#### Hiding future words with causal attention

![causal_attention](./images//3.png)

In [90]:
queries = self_att2.W_query(inputs)
keys = self_att2.W_key (inputs)
values = self_att2.W_value(inputs)

attn_scores = queries @ keys.T # omega
attn_weights  = torch.softmax(
attn_scores / keys.shape[-1]**0.5, dim=-1
)

In [91]:
attn_weights 

tensor([[0.1362, 0.1730, 0.1736, 0.1713, 0.1792, 0.1666],
        [0.1359, 0.1730, 0.1735, 0.1716, 0.1790, 0.1670],
        [0.1366, 0.1729, 0.1734, 0.1714, 0.1788, 0.1669],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.1732, 0.1674],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.1649],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)

In [92]:
masked_att = torch.tril(attn_weights )

In [93]:
masked_att

tensor([[0.1362, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1359, 0.1730, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1366, 0.1729, 0.1734, 0.0000, 0.0000, 0.0000],
        [0.1493, 0.1701, 0.1704, 0.1697, 0.0000, 0.0000],
        [0.1589, 0.1690, 0.1692, 0.1667, 0.1712, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<TrilBackward0>)

In [94]:
# as the row sum is not 1 we need to normalize this again

#? impl -1

# mask = torch.tril(torch.ones_like(attn_scores))  # lower-triangular mask
# masked_att = attn_scores.masked_fill(mask == 0, float('-inf'))
# attn_weights = torch.softmax(masked_att, dim=-1)
# attn_weights

#? impl -2

row_sum =  masked_att.sum(dim=-1 , keepdim=True)
masked_att_norm =  masked_att / row_sum

masked_att_norm


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<DivBackward0>)

In [95]:
# we can do the same with only 3 steps
# current way= att_scores -> normalize ->causal att ->re normalized

# we can do = att_scores -> causal_att -> normalize 

context_length = attn_scores.shape[0]

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[-0.2327,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2396,  0.1015,    -inf,    -inf,    -inf,    -inf],
        [-0.2323,  0.1004,  0.1045,    -inf,    -inf,    -inf],
        [-0.1344,  0.0502,  0.0523,  0.0470,    -inf,    -inf],
        [-0.0349,  0.0520,  0.0538,  0.0331,  0.0708,    -inf],
        [-0.2142,  0.0650,  0.0679,  0.0668,  0.1004,  0.0395]],
       grad_fn=<MaskedFillBackward0>)


In [96]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4400, 0.5600, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2830, 0.3580, 0.3590, 0.0000, 0.0000, 0.0000],
        [0.2264, 0.2579, 0.2583, 0.2574, 0.0000, 0.0000],
        [0.1903, 0.2024, 0.2026, 0.1997, 0.2051, 0.0000],
        [0.1408, 0.1715, 0.1718, 0.1717, 0.1758, 0.1684]],
       grad_fn=<SoftmaxBackward0>)


Masking additional attention weights with dropout

![dropout](./images/4.png)

In [97]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

In [98]:
example = torch.ones(6,6)
layer(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

In [99]:
layer(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7160, 0.7181, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5167, 0.5147, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.4101, 0.0000],
        [0.2815, 0.3430, 0.0000, 0.3434, 0.3516, 0.3368]],
       grad_fn=<MulBackward0>)

Implementing a compact causal self-attention class

In [100]:
# create the data batch
batch = torch.stack((inputs , inputs) , dim=0 )
print(batch)

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])


In [108]:
# Here is a more optimal version from using Linear Layers
class CausalAttention(nn.Module):
    def __init__(self , d_in ,d_out ,  dropout_ratio, context_length, qkv_bias=False ):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out , bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out , bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out , bias=qkv_bias)
        # add the dropout layer
        self.dropout = nn.Dropout(dropout_ratio)
        # causal mask(for the future tokens)
        self.register_buffer('mask' ,torch.triu(torch.ones(context_length , context_length), diagonal=1))


    def forward(self, x):
        b, num_tokens, d_in = x.shape # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result in errors
        # in the mask creation further below.
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method. 
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # ? Here we are taking the transpose form a batch
        # swaps the last two dimensions
        # For a 3D tensor keys with shape [batch_size, seq_len, d_k], this results in [batch_size, d_k, seq_len]

        attn_scores = queries @ keys.transpose(1, 2) 

        # causal mask
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        
        # norm
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        # dropout
        attn_weights = self.dropout(attn_weights) # New

        context_vec = attn_weights @ values
        return context_vec




In [109]:
torch.manual_seed(123)
context_length = batch.shape[1]

ca =  CausalAttention(d_in , d_out, 0.0 ,context_length)
ca(batch)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)

Extending single-head attention to multi-head attention

![multiheaded Attention](./images/5.png)

In [113]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout_ratio, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out,dropout_ratio , context_length, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


Implementing multi-head attention with weight splits

In [115]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


![main map](./images/diagram-4x.png)